In [1]:
!pip install -qU langchain langchain-community langchain-openai faiss-cpu

In [ ]:
pip install pypdf

In [2]:
from pathlib import Path
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import OpenAI

dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)
 
api_key = os.getenv("OPENAI_API_KEY", "").strip()
 
if not api_key:
    raise RuntimeError("OPENAI_API_KEY não foi encontrado no arquivo .env.")
 
if not api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY não é válido. Certifique-se de que a chave começa com 'sk-'.")

print("Chave de API encontrada")

Chave de API encontrada


In [3]:
#import os
#import getpass
#from dotenv import load_dotenv
#load_dotenv()

#from google.colab import userdata

#api_key = userdata.get('OPENAI_API_KEY')

# download das fontes de informação
from langchain_community.document_loaders import PyPDFLoader

doc1 = PyPDFLoader("politica_troca_e_devolucao_geral.pdf").load()
doc2 = PyPDFLoader("politica_reembolso_geral.pdf").load()

documento = doc1 + doc2


C:\Users\marci\AppData\Local\Temp\ipykernel_17304\2415367422.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(openai_api_key=api_key)

embeddings_model.model

'text-embedding-ada-002'

In [5]:
# quebrar os dados em chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

pedacos = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
).split_documents(documento)

In [6]:
len(pedacos)

18

In [7]:
pedacos[:2]

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-21T20:40:08-04:00', 'author': 'Modelo neutro', 'keywords': '', 'moddate': '2026-09-21T20:40:08-04:00', 'subject': 'Documento de exemplo para exercício', 'title': 'Política de Troca e Devolução - Modelo Geral', 'trapped': '/False', 'source': 'politica_troca_e_devolucao_geral.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Modelo neutro para exercício | Versão 1.0 | 22/09/2026\nPágina 1\n POLÍTICA INSTITUCIONAL • MODELO NEUTRO\nPolítica de\nTroca e Devolução\nDiretrizes gerais para troca, devolução, arrependimento e tratamento de\nprodutos com vício.\nDOCUMENTO DE EXEMPLO | MODELO GERAL\n 7 DIAS\n 30 / 90\n 30 DIAS\nArrependimento em compra fora da\n loja\n Dias para reclamar: não durável /\n durável\n Prazo usual para sanar o vício\nComo utilizar este modelo\nSubstitua os campos de identificação e atendimento pelos dados da organização. Ajuste

In [8]:
embeddings_model.embed_query(pedacos[0].page_content)

[-0.008290275000035763,
 0.014357156120240688,
 -0.006829478312283754,
 -0.024023495614528656,
 -0.015862014144659042,
 0.012594710104167461,
 0.011767716147005558,
 -0.006178728770464659,
 -0.007761541288346052,
 -0.015712883323431015,
 0.01159147173166275,
 0.007558181881904602,
 -0.003091059159487486,
 -0.007503952831029892,
 -0.00851397030055523,
 0.013157336972653866,
 0.017692245543003082,
 -0.01935979165136814,
 0.01601114496588707,
 -0.0005681346519850194,
 0.0015167203964665532,
 0.021813658997416496,
 -0.017814261838793755,
 0.0034452429972589016,
 0.013455597683787346,
 0.0006168560939840972,
 0.025108076632022858,
 -0.015834899619221687,
 0.041539497673511505,
 -0.02200346067547798,
 0.01952247880399227,
 -0.0030622498597949743,
 -0.008446183986961842,
 -0.018885286524891853,
 0.012567595578730106,
 -0.0009168108808808029,
 -0.0029995476361364126,
 0.01865481398999691,
 0.017529558390378952,
 -0.009103711694478989,
 0.0012091395910829306,
 0.0172448568046093,
 0.00808691605

In [9]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    documents=pedacos, embedding=embeddings_model
)

In [10]:
from re import search
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [11]:
retriever.invoke("Arrependimento válido")

[Document(id='2cb75150-2a46-4ea5-9b3e-16777eaee724', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-21T20:40:08-04:00', 'author': 'Modelo neutro', 'keywords': '', 'moddate': '2026-09-21T20:40:08-04:00', 'subject': 'Documento de exemplo para exercício', 'title': 'Política de Reembolso - Modelo Geral', 'trapped': '/False', 'source': 'politica_reembolso_geral.pdf', 'total_pages': 5, 'page': 3, 'page_label': '4'}, page_content='e seguro, sem retenção indevida de valores.'),
 Document(id='8101dfcd-22a1-498a-8937-ae32399bd0bf', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-21T20:40:08-04:00', 'author': 'Modelo neutro', 'keywords': '', 'moddate': '2026-09-21T20:40:08-04:00', 'subject': 'Documento de exemplo para exercício', 'title': 'Política de Reembolso - Modelo Geral', 'trapped': '/False', 'source': 'politica_reembolso_geral.pdf', 'total_pages': 5, 'page'

In [12]:
query = "Qual o valor do reembolso?"

In [13]:
query_embed = embeddings_model.embed_query(query)

In [14]:
query_embed

[-0.005131741054356098,
 0.008732105605304241,
 0.01683862693607807,
 -0.013541280291974545,
 -0.027994869276881218,
 0.007937093265354633,
 -0.012844014912843704,
 -0.013358818367123604,
 -0.01378890685737133,
 0.0163433738052845,
 0.010615373030304909,
 0.0025837910361588,
 -0.0049492791295051575,
 -0.011351737193763256,
 -0.0008797270711511374,
 -0.002038034377619624,
 0.022182155400514603,
 -0.006953102070838213,
 0.00874513853341341,
 -0.0032908308785408735,
 -0.0010361230233684182,
 -0.00759823527187109,
 -0.02935030125081539,
 0.006239545531570911,
 -0.0008992765215225518,
 -0.013300170190632343,
 0.014570886269211769,
 0.0013937157345935702,
 0.0567195862531662,
 -0.022051824256777763,
 0.03464169800281525,
 0.0002679909230209887,
 -0.027525682002305984,
 -0.014284160919487476,
 -0.022507980465888977,
 0.0006471695960499346,
 -0.0015435951063409448,
 0.01099332980811596,
 0.020253270864486694,
 0.002937310840934515,
 0.025740161538124084,
 0.022690441459417343,
 0.0230423323810

In [15]:
similar_chunks = retriever.invoke(query)
similar_chunks

[Document(id='9e4351f3-94a8-4d60-b74a-226ad33b833b', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-21T20:40:08-04:00', 'author': 'Modelo neutro', 'keywords': '', 'moddate': '2026-09-21T20:40:08-04:00', 'subject': 'Documento de exemplo para exercício', 'title': 'Política de Reembolso - Modelo Geral', 'trapped': '/False', 'source': 'politica_reembolso_geral.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='POLÍTICA DE REEMBOLSO\nModelo neutro para exercício | Versão 1.0 | 22/09/2026\nPágina 2\n1. Objetivo e aplicação\n1.1 Objetivo\nEsta Política estabelece critérios gerais para restituição de valores pagos pelo consumidor, incluindo\nestorno, devolução por Pix, crédito em conta ou outro meio compatível com a forma de pagamento\nutilizada.\n1.2 Abrangência\nAplica-se a compras de produtos ou serviços quando houver direito ao reembolso previsto em lei, na oferta,\nno contrato ou em condição comercial ex

Resgatar os documentos similares no Banco

In [ ]:
import textwrap
similar_texts = [pedaco.page_content for pedaco in similar_chunks]

#similar_texts
print(textwrap.fill(similar_texts, width=80))

['POLÍTICA DE REEMBOLSO\nModelo neutro para exercício | Versão 1.0 | 22/09/2026\nPágina 2\n1. Objetivo e aplicação\n1.1 Objetivo\nEsta Política estabelece critérios gerais para restituição de valores pagos pelo consumidor, incluindo\nestorno, devolução por Pix, crédito em conta ou outro meio compatível com a forma de pagamento\nutilizada.\n1.2 Abrangência\nAplica-se a compras de produtos ou serviços quando houver direito ao reembolso previsto em lei, na oferta,\nno contrato ou em condição comercial expressamente divulgada pelo fornecedor.\n2. Situações que podem gerar reembolso\nSituação\nTratamento geral\nArrependimento válido\nRestituição integral dos valores pagos, incluindo despesas vinculadas à\ncompra e à devolução, quando aplicável.\nVício não solucionado\nApós o prazo legal, o consumidor pode escolher a restituição imediata do\nvalor pago, atualizado.\nCancelamento pelo fornecedor\nRestituição integral quando a compra não puder ser cumprida, sem\nprejuízo de outros direitos apl

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_consulta_seguro = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto: \n{contexto}"),
        ("human", "{query}")
    ]
)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

modelo = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.5,
    api_key=api_key
)

def no_rag(pergunta):
  return modelo.invoke(pergunta)

def rag(pergunta):
  cadeia = prompt_consulta_seguro | modelo | StrOutputParser()
  # retrieve
  trechos = retriever.invoke(query)
  # augment
  contexto = "\n\n".join(um_trecho.page_content for um_trecho in trechos)
  # generate
  return cadeia.invoke({ "query": pergunta, "contexto": contexto})

In [ ]:
query

In [ ]:
import textwrap
print(textwrap.fill(no_rag(query).content, width=80))

In [ ]:
import textwrap
#rag(query)
print(textwrap.fill(rag(query), width=80))
